In [1]:
from predictors.score_predictor import WorldCupScorePredictor
import torch
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [2]:
config = {
    "model": {
        "input_dim": 2,
        "hidden_dims": [16],
        "activation": "ReLU",
        "target_cols": ["team_1_score", "team_2_score"]
    },
    "training": {
        "epochs": 200,
        "test_year": 2014,
        "loss": {
            "core_loss_fn": "MSELoss",
            "goal_difference_weight": 0.2,
        },
        "data_loaders": {
            "eval": {
                "batch_size": None,
                "shuffle": False,
                "drop_last": False
            },
            "train": {
                "batch_size": 32,
                "shuffle": True,
                "drop_last": False
            }
        },
        "optimiser": {
            "optimiser": "AdamW",
            "learning_rate": 1e-4,
            "weight_decay": 1e-6
        }
    }
}

In [3]:
model = WorldCupScorePredictor(config)

In [4]:
x = torch.Tensor([[1, 2], [-4, 5], [2, -6]])
y = torch.Tensor([[2, 2], [3, 3], [4, 4]])

In [5]:
y_pred_tensor = model(torch.Tensor(x))

In [6]:
y_pred_tensor

tensor([[1.2975, 0.3644],
        [1.3160, 0.2133],
        [0.5896, 0.5799]], grad_fn=<SoftplusBackward0>)

In [7]:
y_pred_tensor[:, 0] - y_pred_tensor[:, 1]

tensor([0.9331, 1.1027, 0.0097], grad_fn=<SubBackward0>)

In [8]:
from predictors.loss import Loss

In [9]:
loss_fn = Loss(config, model.__class__.__name__)

In [10]:
loss = loss_fn(y, y_pred_tensor)
loss.item()

12.504959106445312

In [11]:
dataset = TensorDataset(x, y)
print(dataset[0])
print(dataset[:, 0])

(tensor([1., 2.]), tensor([2., 2.]))
(tensor([ 1., -4.,  2.]), tensor([2., 3., 4.]))


In [12]:
df = pd.read_csv("preprocessed_data/dataset.csv")
df.columns

Index(['year', 'is_knockout', 'team_1_score', 'team_2_score', 'team_1_name',
       'team_1_is_home_nation', 'team_1_top_goalscorers_count',
       'team_1_premier_league_players', 'team_1_laliga_players',
       'team_1_serie_a_players', 'team_1_bundesliga_players',
       'team_1_ligue_1_players',
       'team_1_players_finishing_top_three_europe_domestic',
       'team_1_n_unique_clubs', 'team_1_made_r16_last_time',
       'team_1_made_quarters_last_time', 'team_1_made_semis_last_time',
       'team_1_made_final_last_time', 'team_1_ranking', 'team_2_name',
       'team_2_is_home_nation', 'team_2_top_goalscorers_count',
       'team_2_premier_league_players', 'team_2_laliga_players',
       'team_2_serie_a_players', 'team_2_bundesliga_players',
       'team_2_ligue_1_players',
       'team_2_players_finishing_top_three_europe_domestic',
       'team_2_n_unique_clubs', 'team_2_made_r16_last_time',
       'team_2_made_quarters_last_time', 'team_2_made_semis_last_time',
       'team_2_m

In [13]:
X = df.head()[["team_1_top_goalscorers_count", "team_1_laliga_players"]]
y = df.head()["team_1_score"]
X

,team_1_top_goalscorers_count,team_1_laliga_players
0,2,3
1,0,4
2,0,0
3,2,3
4,0,0


In [14]:
y

0    2
1    2
2    1
3    3
4    0
Name: team_1_score, dtype: int64

In [15]:
y.values.tolist()

[2, 2, 1, 3, 0]

In [16]:
from predictors.goal_probability_predictor import WorldCupGoalsPredictor

In [17]:
goals_cfg = {
    "model": {
        "hidden_dims": [16],
        "activation": "ReLU",
        "exclude_cols": ["year", "is_knockout", "target_team", "opposition"],
        "target_cols": ["team_1_score", "team_2_score"],
        "max_goals": 6,
        "poisson_probabilities": False, # or false for softmax probabilities
        "input_dim": 2
    },
    "training": {
        "epochs": 200,
        "test_year": 2014,
        "loss": {
            "core_loss_fn": "CrossEntropyLoss",
        },
        "dataloaders": {
            "eval": {
                "batch_size": None,
                "shuffle": False,
                "drop_last": False
            },
            "train": {
                "batch_size": 32,
                "shuffle": True,
                "drop_last": False
            }
        },
        "optimiser": {
            "optimiser": "AdamW",
            "lr": 5e-4,
            "weight_decay": 1e-6
        }
    }
}



In [18]:
x = torch.Tensor([[1, 2], [-4, 5], [2, -6]])
y = torch.Tensor([[0, 0, 0, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0]])

model = WorldCupGoalsPredictor(goals_cfg)

In [32]:
z = model(x, softmax_output=False)
z[0]

tensor([ 0.1555,  0.1472, -0.0228,  0.1825,  0.7121, -0.3955, -0.2275],
       grad_fn=<SelectBackward0>)

In [20]:
loss = Loss(goals_cfg, model.__class__.__name__)

In [21]:
loss.loss_fn(z, y)

tensor(2.3729, grad_fn=<DivBackward1>)

In [22]:
from predictors import Trainer

In [23]:
tensors = {}
tensors["x_train"] = x
tensors["y_train"] = y
tensors["x_test"] = x
tensors["y_test"] = y
trainer = Trainer(model, goals_cfg, tensors = tensors)

In [31]:
import numpy as np
trainer.run_epoch(1, np.inf)
z = model(x, softmax_output=True)
print(trainer.history["eval_loss"])
print(z)
print(y)

[np.float64(2.372915506362915), np.float64(2.3570384979248047), np.float64(2.341796398162842), np.float64(2.3268063068389893), np.float64(2.311995267868042), np.float64(2.2971513271331787), np.float64(2.2822482585906982), np.float64(2.2672693729400635)]
tensor([[0.1458, 0.1446, 0.1220, 0.1498, 0.2544, 0.0840, 0.0994],
        [0.0333, 0.0900, 0.0607, 0.0611, 0.6632, 0.0295, 0.0623],
        [0.2102, 0.0082, 0.0061, 0.0723, 0.3839, 0.2334, 0.0860]],
       grad_fn=<SoftmaxBackward0>)
tensor([[0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0.]])
